# 02 · Calidad de datos y limpieza

**Objetivo.** Auditar la calidad variable por variable, verificar la lógica de saltos contra los datos y limpiar los códigos especiales **sin recodificar por convención**.

| | |
|---|---|
| **Entradas** | DataFrame unido del notebook 01 |
| **Salidas** | `tabla_01_calidad_datos`, `tabla_02_verificacion_saltos`, `tabla_03_inventario_codigos_8_9` |

> Proyecto: *Modelamiento epidemiológico de la depresión en la población peruana de 15 y
> más años (ENDES 2025)* · Epidemiología 2026-2 · UNMSM


In [ ]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
sys.path.insert(0, str(Path.cwd().parents[1] / "codigo" / "src"))
sys.path.insert(0, str(Path.cwd().parents[1] / "codigo" / "scripts"))

import numpy as np
import pandas as pd
from depresion import config as cfg, io as dio, viz
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)
viz.aplicar_tema()
print("Proyecto:", cfg.RAIZ.name)

In [ ]:
from depresion import limpieza as lim
from construir_dataset import auditoria_calidad
csalud, meta_cs = dio.leer_sav(cfg.ARCHIVO_CSALUD)
rech0, meta_r0 = dio.leer_sav(cfg.ARCHIVO_RECH0)
meta = meta_cs.combinar(meta_r0)
df = lim.unir_bases(csalud, rech0)

## Verificación de la lógica de saltos

Una *violación* es un caso con dato presente sin cumplir el filtro que debería habilitarlo. Cero violaciones significa que el `NaN` es «no aplica», no «no respondió» — y por tanto **no se imputa**.

In [ ]:
saltos = lim.verificar_saltos(df)
saltos[["cadena", "filtro", "variable", "esperados", "presentes", "violaciones", "veredicto"]]

## La trampa de los códigos 8 y 9

La regla habitual dice que `8 = No sabe` y `9 = Rehúsa`. Es cierta para la mayoría de
variables, pero **no para todas**, y aplicarla a ciegas corrompe los datos.

In [ ]:
inv = lim.inventariar_codigos_especiales(df, meta)
print(f"Variables con código 8 o 9: {len(inv)}")
print("\nExcepciones — donde el código NO significa dato faltante:")
inv[inv.decision.str.startswith("REVISAR")][
    ["variable", "descripcion", "etiqueta_8", "etiqueta_9"]].head(10)

### El caso de `QS711`

El diccionario del INEI declara los códigos 1, 2, 8 y 9. Los datos graban 1, 2, 3 y 4. Es un **error de etiquetado en la fuente oficial**.

In [ ]:
print("QS710 — datos: ", sorted(df.QS710.dropna().unique()))
print("QS710 — etiquetas:", meta.valores("QS710"))
print("\nQS711 — datos: ", sorted(df.QS711.dropna().unique()))
print("QS711 — etiquetas:", meta.valores("QS711"))
print(f"\n¿Existen los códigos 8 o 9 en QS711? {df.QS711.isin([8, 9]).any()}")
print(f"Casos en las categorías severas (3, 4): {int(df.QS711.isin([3, 4]).sum())}")

In [ ]:
df, informe_qs711 = lim.corregir_qs711(df)
df, bitacora = lim.limpiar_codigos_faltantes(df, meta)
df, antro = lim.limpiar_antropometria(df)
print(bitacora.accion.value_counts().to_string())
antro

## Conclusiones

- Las **siete cadenas de saltos** son estructuralmente limpias: cero violaciones.
- De 127 variables con código 8 o 9, se limpiaron **87** y quedaron **40 intactas**.
  Entre las intactas: `QSINTM` y `QS22M` (8 = agosto, 9 = septiembre), `QS25AA`
  (lengua materna) y variables de conteo donde 8 y 9 son cantidades reales.
- `QS711` conserva su escala 1-4 y sus 33 casos severos.
- El umbral de no-medición (`≥ 900`) se aplica **solo** a peso, talla y perímetro
  abdominal. La presión arterial no usa ese código.
